# Price Predictor

This mini project automates the process of finding, evaluating, and notifying users about the best online product deals by leveraging AI agents and tool integrations.

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
import chromadb
import logging
from agents.items import Item
from tqdm import tqdm
from agents.automating_agent import AutoPlanningAgent
MODEL = "gpt-5.1"
DB = "products_vectorstore"
LITE_MODE = True

In [ ]:
load_dotenv(override=True)

In [ ]:
hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
client = chromadb.PersistentClient(path=DB)

In [ ]:
collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
agent = AutoPlanningAgent(collection)

In [ ]:
agent.plan()